In [25]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [21]:
# Used to create token ids, encode data, and decode tokens
class Processor:
    def __init__(self):
        self.encodings = {}
        self.decodings = {}

    # Read data to create token ids
    def ingest(self, data=str):
        raw_chars = list(data)
        unique_chars = sorted(list(set(raw_chars)))

        # Assign each char to a token id
        for token_id, u_char in enumerate(unique_chars):
            self.encodings[u_char]    = token_id
            self.decodings[token_id] = u_char

        # Assign vocab size
        self.vocab_size = len(unique_chars)

    # Encode characters to token ids
    def encode(self, data=str):
        raw_chars = list(data)
        tokens = [self.encodings[raw_char] for raw_char in raw_chars]
        return tokens
    
    # Decode tokens into characters
    def decode(self, data=list):
        decoded_tokens = [self.decodings[token_id] for token_id in data]
        decoded_string = "".join(decoded_tokens)
        return decoded_string

In [23]:
processor = Processor()

# Read and ingest data
with open("Data/tiny-shakespeare.txt", 'r') as f:
  data = f.read()
processor.ingest(data)
tokenized_data_list = processor.encode(data)

# Convert into tensor
tokenized_data = torch.tensor(tokenized_data_list, dtype=torch.long)

# Get vocab size
vocab_size = processor.vocab_size

In [ ]:
# Creates instance of one model
# All hyperparameters and training are done inside this object
class Model():
    def __init__(self, vocab_size, B=32, T=64, C=128, H=128):
        # Define hyper parameters
        self.B = B  # Batch size
        self.T = T  # Sequence length or Block size
        self.C = C  # Embedding dimension
        self.H = H  # Matches C as we only are implementing one head
        self.vocab_size = vocab_size

        # Define matrices
        self.embedding_matrix = torch.randn(vocab_size, C)   # Holds embedding vectors for each token (Vocab_Size x C)
        self.W_q = torch.randn(C, H)     # Holds the Query weights (What we look for given input)
        self.W_k = torch.randn(C, H)     # Holds the Key weights (What the input holds/represents or has to offer)
        self.W_v = torch.randn(C, H)     # Holds the Value weights (Content that should be passed forward)
        ### Since our Head size is the same as our Embedding dimension, we can use the embedding matrix as our lm_head matrix

    # Create (B, T, C) matrix of randomly selected tokens from given data
    def build_train_batch(self, data):
        indices = torch.randint(len(data) - self.T, (self.B,))
        x = torch.stack([data[ix:ix+self.T] for ix in indices])
        y = torch.stack([data[ix+1:ix+self.T+1] for ix in indices])
        return x, y
        
    # Perform one forward pass to calculate predicted output
    def forward_pass(self, X):   # X is expected (B, T, C)
        return



In [46]:
model = Model(vocab_size)

x, y = model.build_train_batch(tokenized_data)
torch.set_printoptions(profile="full")
print(f"{x[0]}\n{y[0]}")

tensor([ 1, 61, 47, 50, 50,  1, 58, 53,  1, 46, 47, 51,  1, 39, 52, 42,  1, 54,
        50, 59, 41, 49,  1, 53, 59, 58,  1, 46, 47, 57,  1, 43, 63, 43, 57,  2,
         0,  0, 16, 33, 23, 17,  1, 34, 21, 26, 15, 17, 26, 32, 21, 27, 10,  0,
        37, 53, 59,  1, 57, 46, 39, 50, 50,  1])
tensor([61, 47, 50, 50,  1, 58, 53,  1, 46, 47, 51,  1, 39, 52, 42,  1, 54, 50,
        59, 41, 49,  1, 53, 59, 58,  1, 46, 47, 57,  1, 43, 63, 43, 57,  2,  0,
         0, 16, 33, 23, 17,  1, 34, 21, 26, 15, 17, 26, 32, 21, 27, 10,  0, 37,
        53, 59,  1, 57, 46, 39, 50, 50,  1, 52])
